<a href="https://colab.research.google.com/github/Fizzah-Amir14/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fizzah-Amir14/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb pandas scikit-learn matplotlib

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 **— "What Predicts Health?" (ML Appendix, Random Forest). The paper reports Random Forest feature importance for predicting Health Score: Average Position (43%), Impressions (32%), and Scroll Depth (15%) — roughly 90% of importance combined.

Where does the label come from? Health Score is explicitly defined earlier in the paper as Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts) — so the label is arithmetically built from three of the same signals the model calls its top predictors.

My methodology question: does a model "predicting" a composite score partly out of its own ingredients tell us anything beyond confirming the composite formula is internally consistent? The paper itself flags this ("does not imply external causation") — the open question is whether "feature importance" is even the right frame here, versus describing it as construction weight.

**Finding 2 — "What Predicts Growth?"** (ML Appendix, Logistic Regression, 71% holdout accuracy). The paper reports Content Age, Days Since Update, and Days Visible as the strongest signals separating growing from declining pages.

Where does the label come from? The paper defines Trend Direction (the growth/decline label) as "calculated from 30d-vs-prev-30d impression change."

**My methodology question:** were any input features — particularly "Days Visible" or a recent-impressions signal — close enough to that same 30-day-vs-previous-30-day window to be near-restatements of the label rather than independent predictors? Was the 71% holdout accuracy measured on a random split or a time-aware split, since a random split would let a near-duplicate-of-label feature inflate the score without adding real decision value? (This is the same failure mode I check for in my own model in Section 3 — my impressions_last_30d / impressions_prev_30d features dominate importance the same way, and I flag it there.)

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 (ML-08) model evaluated Random Forest and Gradient Boosting against a Logistic Regression baseline for CTR/engagement decay. The permutation importance table showed
impressions_last_30d and impressions_prev_30d together accounting for ~86% of importance, with almost every other feature near zero - a strong signal the "before" split may be leaking client-
specific patterns rather than learning a generalizable rule.

Honest split choice: grouped by client_id. The dataset has no explicit date/timestamp column exposed in the schema, but it does have client_id - and a plain random split lets the same client's
pages appear in both train and test, so the model can partly memorize client-specific quirks instead of learning transferable signal. GroupKFold on client_id closes that gap directly.

In [ ]:
!git clone https://github.com/Fizzah-Amir14/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 204, done.
remote: Counting objects: 100% (204/204), done.
remote: Compressing objects: 100% (156/156), done.
remote: Total 204 (delta 98), reused 102 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (204/204), 1.93 MiB | 9.88 MiB/s, done.
Resolving deltas: 100% (98/98), done.


In [ ]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, average_precision_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("shape:", df.shape)

# Label: DO NOT use trend_direction / trend_pct as a feature — label-only.
df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

# Candidate feature set — drop IDs, the label source columns, and anything derived from the label.
drop_cols = ["content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"]
feature_cols = [c for c in df.columns if c not in drop_cols]

# Keep numeric features only for this audit pass (tier/type columns are categorical strings —
# note them here rather than silently one-hot-encoding a leak-prone column in a rush).
numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
print(f"{len(numeric_features)} numeric features in play:", numeric_features)

X = df[numeric_features].fillna(0)
y = df["is_declining_label"]
groups = df["client_id"]

shape: (30000, 44)
29 numeric features in play: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


BEFORE - plain random split (what the Week-5 model effectively did)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model_before = RandomForestClassifier(max_depth=6, min_samples_leaf=20, n_estimators=200, random_state=42)
model_before.fit(X_train, y_train)
proba_before = model_before.predict_proba(X_test)[:, 1]
pred_before = model_before.predict(X_test)

metrics_before = {
    "split": "random (before)",
    "roc_auc": roc_auc_score(y_test, proba_before),
    "avg_precision": average_precision_score(y_test, proba_before),
    "precision": precision_score(y_test, pred_before),
    "recall": recall_score(y_test, pred_before),
}
print(metrics_before)

{'split': 'random (before)', 'roc_auc': np.float64(0.820080040320054), 'avg_precision': np.float64(0.8301650884731916), 'precision': 0.7192168985059247, 'recall': 0.8585485854858549}


**AFTER - GroupKFold split by client_id (honest split)**



In [ ]:
gkf = GroupKFold(n_splits=5)
fold_metrics = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    m = RandomForestClassifier(max_depth=6, min_samples_leaf=20, n_estimators=200, random_state=42)
    m.fit(X_tr, y_tr)
    proba = m.predict_proba(X_te)[:, 1]
    pred = m.predict(X_te)

    fold_metrics.append({
        "fold": fold,
        "roc_auc": roc_auc_score(y_te, proba),
        "avg_precision": average_precision_score(y_te, proba),
        "precision": precision_score(y_te, pred, zero_division=0),
        "recall": recall_score(y_te, pred, zero_division=0),
    })

fold_df = pd.DataFrame(fold_metrics)
print(fold_df)

metrics_after = {
    "split": "GroupKFold by client_id (after)",
    "roc_auc": fold_df["roc_auc"].mean(),
    "avg_precision": fold_df["avg_precision"].mean(),
    "precision": fold_df["precision"].mean(),
    "recall": fold_df["recall"].mean(),
}
print("\nMean across folds:", metrics_after)

   fold   roc_auc  avg_precision  precision    recall
0     0  0.710903       0.702612   0.623410  0.699272
1     1  0.739517       0.830231   0.708593  0.951879
2     2  0.851347       0.760175   0.570291  0.951443
3     3  0.752079       0.790256   0.762036  0.928232
4     4  0.799414       0.811969   0.738786  0.915577

Mean across folds: {'split': 'GroupKFold by client_id (after)', 'roc_auc': np.float64(0.7706520507715483), 'avg_precision': np.float64(0.7790486434970278), 'precision': np.float64(0.6806233399815615), 'recall': np.float64(0.889280616857417)}


In [ ]:
comparison = pd.DataFrame([metrics_before, metrics_after]).set_index("split")
print(comparison)
print("\nGap (before - after):")
print(comparison.loc["random (before)"] - comparison.loc["GroupKFold by client_id (after)"])

                                  roc_auc  avg_precision  precision    recall
split                                                                        
random (before)                  0.820080       0.830165   0.719217  0.858549
GroupKFold by client_id (after)  0.770652       0.779049   0.680623  0.889281

Gap (before - after):
roc_auc          0.049428
avg_precision    0.051116
precision        0.038594
recall          -0.030732
dtype: float64


The gap between the random split and the grouped split is real but moderate: ROC-AUC drops from 0.820 (random) to 0.771 (GroupKFold by client_id), and precision drops from 0.72 to 0.68 — roughly a 5-point ROC-AUC gap and 4-point precision gap. Recall actually improves slightly (0.86 → 0.89) under the honest split, so this isn't total collapse, but the consistent drop across ROC-AUC, average precision, and precision confirms the random split was inflating performance by letting the model see patterns from the same client in both train and test. The honest numbers (mean ROC-AUC 0.77, precision 0.68, recall 0.89) are what I'd report going forward, not the original 0.82/0.72 pair.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# 1. Explicit check: label-source columns must never be features.
label_source_cols = ["trend_direction", "trend_pct"]
leaked = [c for c in numeric_features if c in label_source_cols]
assert not leaked, f"Leakage detected: {leaked}"
print("No direct label-source columns in the feature set.")

# 2. Correlation check: does any single feature correlate suspiciously highly with the label?
corrs = X.assign(label=y).corr()["label"].drop("label").sort_values(key=abs, ascending=False)
print("\nTop feature correlations with the label:")
print(corrs.head(10))

No direct label-source columns in the feature set.

Top feature correlations with the label:
days_with_impressions     0.190055
content_age_days         -0.163882
age_tier_order           -0.156142
word_count                0.118863
char_count                0.108025
impressions_last_30d     -0.093980
days_since_last_update    0.081383
clicks_last_30d          -0.071935
sessions_last_30d        -0.063842
ctr                      -0.061911
Name: label, dtype: float64


In [ ]:
# 3. Importance concentration check — flagged from the ML-08 permutation importance table:
# impressions_last_30d + impressions_prev_30d together carried ~86% of importance while nearly
# every other feature sat at ~0. Re-check that concentration here on the honest split.
importances = pd.Series(model_before.feature_importances_, index=numeric_features).sort_values(ascending=False)
print("Feature importance concentration (random-split model):")
print(importances.head(10))
top2_share = importances.head(2).sum() / importances.sum()
print(f"\nTop-2 features carry {top2_share:.1%} of total importance.")

if top2_share > 0.7:
    print("\nFLAG: heavy concentration in impressions_last_30d / impressions_prev_30d.")
    print("These two features together approximate an impression TREND (last vs prev 30 days),")
    print("which sits very close to the same information the label (trend_direction) encodes.")
    print("This isn't a direct leak — trend_direction/trend_pct are excluded — but it's a near-proxy:")
    print("the model may be learning 'impressions went down' as a stand-in for the label itself,")
    print("rather than learning the broader CTR/engagement signal the rule in ML-07 was built on.")

Feature importance concentration (random-split model):
impressions_prev_30d     0.354352
impressions_90d          0.102780
impressions_last_30d     0.089980
days_with_impressions    0.086127
avg_position             0.072341
content_age_days         0.063784
age_tier_order           0.036190
clicks_last_30d          0.032061
word_count               0.029441
sessions_last_30d        0.024766
dtype: float64

Top-2 features carry 45.7% of total importance.


On this retrained feature set, importance concentration is less extreme than my original ML-08 run — the top two features (impressions_prev_30d and impressions_90d) carry 45.7% of importance combined, versus ~86% originally, likely because this run drops the categorical tier features and adds impressions_90d. Correlations with the label are all modest (highest is days_with_impressions at r=0.19), so no single feature is an obvious direct leak. That said, impressions_prev_30d and impressions_last_30d are still an impression-trend signal conceptually close to how trend_direction itself is defined, so I'd treat performance gains tied specifically to those two features with the same caution I raised about the paper's own Finding 2 in Section 1.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# Example — rewrite any claim from your ML-08 notebook or capstone writeup that overstates
# what the evidence supports. Fill in your own claims below; this is a template pattern.

claims = [
    {
        "original": "The model predicts which pages will decline.",
        "rewritten": "The model's score is directionally associated with observed decline in "
                      "the grouped-split evaluation (mean ROC-AUC of {:.2f} across 5 client-held-out "
                      "folds) — it is a decision-support signal for prioritizing review, not a "
                      "certain prediction.".format(metrics_after["roc_auc"]),
    },
    {
        "original": "Accuracy of 82.1% proves the model works well.",
        "rewritten": "Under an honest, client-grouped split the model's measured precision was "
                      "{:.2f} and recall was {:.2f} — meaningfully different from a same-client "
                      "random split, and the gap itself is worth disclosing alongside the number."
                      .format(metrics_after["precision"], metrics_after["recall"]),
    },
]
for c in claims:
    print("ORIGINAL: ", c["original"])
    print("REWRITTEN:", c["rewritten"])
    print()

# Add your own real claims from your capstone writeup / ML-08 notebook here, same pattern.

ORIGINAL:  The model predicts which pages will decline.
REWRITTEN: The model's score is directionally associated with observed decline in the grouped-split evaluation (mean ROC-AUC of 0.77 across 5 client-held-out folds) — it is a decision-support signal for prioritizing review, not a certain prediction.

ORIGINAL:  Accuracy of 82.1% proves the model works well.
REWRITTEN: Under an honest, client-grouped split the model's measured precision was 0.68 and recall was 0.89 — meaningfully different from a same-client random split, and the gap itself is worth disclosing alongside the number.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.